# RFP 전처리 전체 과정

원본 추출부터 최종 Elements 검증까지. 새 실행 결과는 별도 경로에 저장.

실행 출력: Notebook별 독립 IPython namespace에서 셀 순서 검증. Jupyter 커널 실행 검증은 별도.

### GitHub 공유본 안내

- 원본 Notebook을 복사하여 정리한 공유본입니다. 저장된 실행 출력과 실행 번호는 제거했습니다. 본문의 검증 수치는 기존 실행 기록이며 이번 업로드에서 재실행한 결과가 아닙니다.
- 개인 PC 경로는 `./local_inputs` 아래의 상대경로로 치환했습니다. 이 경로는 입력 배치 예시이며 실제 실행 전 설정 셀과 외부 입력 경로를 준비해야 합니다.
- 외부 데이터·설정·원본 코드·승인 패킷은 이번 PR에 포함하지 않습니다. 저장소만으로 전체 실행을 재현할 수 없습니다.
- 원문 의미 보존 검증 및 Native부터 최종 결과까지의 연속 재현은 미완료입니다. 진행 중인 검증은 실행하거나 변경하지 않았습니다.


## 1. 경로와 의존성

`BASE`는 기존 9개 Notebook 폴더. 경로를 옮기면 이 셀에서 변경.

- 외부 코드: `original_sources/legacy/src`의 파싱·정제·복구 함수.
- 전체 실행: `replay_initial.py`, `replay_mapping.py`, 최종 패키지의 `adapter/reproduce_elements.py`.
- 외부 데이터: `PROJECT` 원본·검토 기록, `HISTORY` 단계별 결과, `FINAL` 최종 패키지.
- 실행 환경: 기존 `rfp-rag` Python, pandas, IPython, HWP loader, PyMuPDF, pyhwp/hwp5proc. Mapper는 기존 별도 Python과 패키지·토큰 캐시 사용.

경로 설정·출력·비교 함수는 아래 셀에 포함. 데이터 확인 셀은 외부 Python 실행 없이 동작하지만 기존 데이터 파일이 필요하다. 파싱·정제·Elements 생성은 외부 코드가 필요하다.

Mapper 내부의 토큰 수·표 행 유틸리티 의존성은 유지. 청크 생성·분할 함수는 실행하지 않는다.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from itertools import zip_longest
from IPython.display import display
import sys, json, re, copy, os, hashlib, subprocess, uuid, unicodedata
import pandas as pd

# 원본 폴더는 읽기 전용 입력. 새 결과는 현재 작업의 work에 저장한다.
BASE = Path('./local_inputs/Desktop/중급프로젝트/preprocessing')
WORK = Path('./local_inputs/Documents/Codex/2026-10-12/notebook-9-notebook-2-1-01-2/work')
SETTINGS = json.loads((BASE / 'settings.json').read_text())
PROJECT = Path(SETTINGS['project'])
HISTORY = Path(SETTINGS['historical_root'])
FINAL = Path(SETTINGS['final_package'])
LEGACY = BASE / 'original_sources/legacy'
sys.dont_write_bytecode = True
sys.path.insert(0, str(LEGACY / 'src'))
RUN_STATE = json.loads((BASE / 'run_state.json').read_text())

def read(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def jl(path):
    with Path(path).open(encoding='utf-8') as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def clip(value, n=650):
    text = value if isinstance(value, str) else json.dumps(value, ensure_ascii=False, default=str)
    return text if len(text) <= n else text[:n] + ' …'

def frame(rows):
    with pd.option_context('display.max_colwidth', 120, 'display.max_rows', 12, 'display.max_columns', 12):
        display(pd.DataFrame(rows))

def state():
    return RUN_STATE

def save_state(values):
    RUN_STATE.update({k: str(v) for k, v in values.items()})

def unique_output(label):
    path = WORK / 'preprocessing_runs' / f'{label}_{uuid.uuid4().hex[:10]}'
    path.parent.mkdir(parents=True, exist_ok=True)
    assert not path.exists()
    return path

def command(python, script, args, label):
    log = unique_output(label + '.log')
    env = os.environ.copy()
    env['PYTHONDONTWRITEBYTECODE'] = '1'
    with log.open('w') as f:
        result = subprocess.run([str(python), str(script), *map(str, args)], stdout=f, stderr=subprocess.STDOUT, env=env)
    if result.returncode:
        raise RuntimeError(f'실행 실패({result.returncode}): {log}')
    print(log.read_text()[-1800:])
    return log

def data_versions():
    data = next(p for p in PROJECT.iterdir() if unicodedata.normalize('NFC', p.name) == '원본 데이터')
    return data / 'RAG_정제데이터'

def checks_equal_jsonl(a, b):
    # 순서·본문·메타데이터를 포함한 전체 행 비교.
    count = 0
    for count, (x, y) in enumerate(zip_longest(jl(a), jl(b)), 1):
        assert x == y, f'행 {count} 불일치'
    return {'행 수': count, '전체 값 동일': True, '바이트 해시 동일': sha(a) == sha(b)}

print('원본:', PROJECT)
print('새 실행 결과:', WORK / 'preprocessing_runs')

## 2. 데이터 로드

원본 100개, 동일 파일을 묶은 대표 문서 98개.

In [ ]:
from preprocess import inventory, timed_extract, SAMPLE
original_root, documents = inventory(PROJECT)
hwp_row = next(r for r in documents if r['doc_id']==SAMPLE)
pdf_row = next(r for r in documents if r['file_type']=='.pdf')
# 실제 재추출: 이전 JSON 캐시를 읽지 않고 원본 바이너리를 파싱합니다.
representative_raw = timed_extract(original_root / hwp_row['representative_path'], hwp_row)
pdf_raw = timed_extract(original_root / pdf_row['representative_path'], pdf_row)
frame([{'종류':'HWP','추출 요소':len(representative_raw),'형태':dict(Counter(e['element_type'] for e in representative_raw))},
       {'종류':'PDF','추출 요소':len(pdf_raw),'형태':dict(Counter(e['element_type'] for e in pdf_raw))}])
frame([{'순서':e['list_index'],'요소 유형':e['element_type'],'페이지':e['page_number'],'원문 샘플':clip(e['raw_text'])} for e in representative_raw[:4]])
frame([{'페이지':e['page_number'],'이미지 수':e['loader_metadata']['image_count'],'본문 샘플':clip(e['raw_text'])} for e in pdf_raw[:2]])

### HWP/PDF 추출 결과 확인

- HWP: 237개 요소 추출 (본문 1개, 표 226개, 각주 1개, 이미지 9개)
- PDF: 49개 페이지 추출
- HWP는 표가 대부분을 차지하며, 원문에 Markdown 형태의 표 구조가 포함되어 있음
- PDF는 페이지별 텍스트와 이미지 정보를 확인할 수 있음
- 추출된 원문에 줄바꿈(`\n`)과 표 구분자(`|`)가 포함되어 있어 후속 정제 필요

In [ ]:
tables = [e for e in representative_raw if e['element_type']=='table']
frame([{'요소 번호':e['list_index'],'파서 표 원문':clip(e['raw_text'],1000),'loader metadata':clip(e['loader_metadata'],300)} for e in tables[:2]])
print('PDF 첫 페이지 blocks 수:',len(pdf_raw[0]['loader_metadata']['layout_blocks']))
print('원본 해시 확인:',sha(original_root/hwp_row['representative_path']) == hwp_row['sha256'])

### HWP 표 원문 확인

- 표 요소 1, 2번 확인
- 표가 Markdown 형식(`|`, `---`)으로 추출됨
- 표 내부에 빈 셀이 다수 포함되어 있음
- 파일명 등 원본 문서 정보는 metadata에 유지됨
- 불필요한 빈 셀과 표 구조는 후속 전처리에서 확인 필요

## 4. 전체 정제·정규화

기존 inventory → timed_extract → process → finalize 실행. raw/clean/search 변화 확인.

In [ ]:
REEXTRACT_ALL = True  # True면 시간이 걸리는 원본 전체 재실행
if REEXTRACT_ALL:
    destination = unique_output('01_raw_clean')
    command(SETTINGS['legacy_python'], BASE/'replay_initial.py', ['--output',destination], '01_run')
    command(SETTINGS['legacy_python'], LEGACY/'src/finalize.py', ['--project',PROJECT,'--output',destination], '01_finalize')
    save_state({'initial':destination})
initial = Path(state()['initial'])
frame([read(initial/'summary.json')])
frame([read(initial/'전체_검증결과.json')])
# 각 record의 raw_start:raw_end가 원본 요소의 raw_text와 같아야 합니다.
print('처리 실패:',sum(r['status']=='처리 실패' for r in read(initial/'document_reports.json')))

### 전체 전처리 파이프라인 실행

기존에 단계별로 진행했던 전처리 작업을 하나의 파이프라인으로 실행

1. **Inventory**: 원본 HWP/PDF 파일 목록 확인 및 중복 문서 구분
2. **Timed Extract**: 원본 문서에서 텍스트, 표 등 요소 추출
3. **Process**: 추출 데이터 정제 및 정규화, RAG 레코드 생성
4. **Finalize**: 처리 결과 통합 및 최종 데이터 검증

**데이터 처리 흐름**
- Raw: 원본 문서에서 추출한 데이터
- Clean: 공백, 문자 및 형식 등을 정리한 데이터
- Search: 검색에 사용할 수 있도록 가공한 데이터

기존에는 각 단계를 개별 실행했지만, 현재는 전체 과정을 연결하여 실행하도록 구성함.

### 전체 데이터 정제 및 정규화 결과

- 원본 문서 100개 중 중복 제외 98개 처리
- 정제된 RAG 레코드 22,435개 생성
- 원본 요소 13,010개 확인
- 문서 처리 실패 0건
- 정제 및 구조 변경 17,202건
- 예외 항목 6,115건 (690개 그룹)
- 원본 데이터 완전성 확인은 아직 완료되지 않음

전체 정제 및 정규화 검증은 통과했지만, 예외 항목과 원본 데이터 누락 여부는 추가 확인 필요

In [ ]:
from preprocess import inventory, process, SAMPLE
original_root, documents = inventory(PROJECT)
row = next(r for r in documents if r['doc_id']==SAMPLE)
initial = Path(state()['initial'])
raw = read(initial/'raw'/f"{row['doc_id']}.json")
# 실제 process 호출. 학습 출력만 추가하고 원본 함수는 변경하지 않습니다.
units, changes, exceptions, links, report = process(row,raw,PROJECT/'artifacts/preprocessing')
changed = [u for u in units if u['raw_text']!=u['clean_text'] or u['clean_text']!=u['search_text']]
frame([{'record_id':u['record_id'],'정제 전':clip(u['raw_text']),'정제 후':clip(u['clean_text']),'검색 표현':clip(u['search_text']),'요구사항 ID':u['requirement_ids']} for u in changed[:3]])
frame([{'규칙':c.get('rule',c.get('operation')),'좌표 공간':c.get('coordinate_space'),'전':clip(c.get('original',c.get('before'))),'후':clip(c.get('replacement',c.get('after')))} for c in changes[:5]])

### 텍스트 정제 전후 비교

대표 문서를 기준으로 정제 전후 텍스트와 변경 이력 확인

- 표의 빈 행과 불필요한 구분자 제거
- 승인된 앞쪽 목차 제거
- 검색 표현에 남아 있는 문자 깨짐 확인
- 정제 전 원문과 정제 후 텍스트를 비교하여 변경 내용 확인
- 변경 이력에 적용 규칙, 좌표 공간, 변경 전후 값 기록

일부 목차에서 깨진 문자가 확인되어 추가 정제 필요

## 5. 문자 깨짐 복구

HWP 탭 제어정보와 승인된 PDF 글리프 근거로 복구. 이어서 예외 재검사.

In [ ]:
REPAIR_NEW = True  # 02에서 새 전체 추출을 실행했다면 True로 바꿉니다.
if REPAIR_NEW:
    repaired = unique_output('03_repaired')
    command(SETTINGS['legacy_python'],LEGACY/'src/repair.py',['--project',PROJECT,'--input',state()['initial'],'--output',repaired],'03_repair')
    rechecked = unique_output('03_rechecked')
    command(SETTINGS['legacy_python'],LEGACY/'src/recheck.py',['--project',PROJECT,'--input',repaired,'--baseline',state()['initial'],'--output',rechecked],'03_recheck')
    save_state({'repaired':repaired,'rechecked':rechecked})
repaired = Path(state()['repaired'])
repair_changes = list(jl(repaired/'문자복구_변경내역.jsonl'))
frame([{k:clip(v,500) for k,v in r.items() if k in ['record_id','before','after','rule','checks','원본','복구']} for r in repair_changes[:3]])
print('이번 전체 복구 실행 변경 기록:',len(repair_changes))
# 실제 JSON 구조도 1개 확인합니다. 모든 값 출력 대신 key와 대표 값만 봅니다.
frame([{'필드':k,'실제 값':clip(v,500)} for k,v in repair_changes[0].items()][:12])
recheck_dir=Path(state()['rechecked'])
frame([read(recheck_dir/'예외_재검사_검증.json')])

### 문자 깨짐 복구

HWP 추출 과정에서 목차 번호 주변에 잘못 들어간 문자와 비정상적인 표현 확인

- 복구 대상 391개 레코드
- 총 4,168건의 문자 보정
- 복구 대상 391건 모두 처리 완료
- 수정 전후 텍스트 비교

## 6. 표 구조 개선

승인된 3개 위치를 원본 좌표·병합 정보로 재직렬화. 변경 원장의 결과와 비교.

In [ ]:
from preprocess import inventory
from structure_source import HWP5Reader, table_grid
from repair_confirmed_tables import 원본직렬화, 표식보존
original_root, documents=inventory(PROJECT)
bydoc={r['doc_id']:r for r in documents}
ledger_path=data_versions()/'version_04_tables_20261007T021326_76e75bc3/표복구_변경내역.json'
ledger=read(ledger_path)
actual=[];grids=[]
for entry in ledger:
    # 실제 과거 변경 후 구조의 원본 주소를 사용합니다. 새 주소를 추측하지 않습니다.
    structure=entry['변경']['table_structure']['후']
    path=original_root/bydoc[entry['doc_id']]['representative_path']
    assert sha(path)==entry['doc_id']
    reader=HWP5Reader(path)
    records=list(reader._parse_records(reader._read_section(structure['section'])))
    grid=table_grid(records,structure['root_table_record'],reader)
    assert grid is not None and grid['source_table_sha256']==structure['source_table_sha256']
    rendered,new_structure,relations=원본직렬화(reader,records,grid)
    reader.close()
    before=entry['변경']['clean_text']['전'];expected=entry['변경']['clean_text']['후']
    rendered=표식보존(rendered,before)
    assert rendered==expected, '과거 복구 후 텍스트와 재실행 결과 불일치'
    assert len(new_structure['consumed_paragraph_records'])==len(set(new_structure['consumed_paragraph_records']))
    actual.append({'record_id':entry['record_id'],'개선 전':clip(before,1400),'원본 재직렬화 후':clip(rendered,1400),'과거 결과 동일':True,'하위 표 관계':len(relations)})
    grids.extend([{'문서':entry['doc_id'][:12], 'table_record':t['source_table_record'],'row':c['row'],'column':c['column'],'rowspan':c['rowspan'],'colspan':c['colspan'],'셀 원문':clip(c['text'],250)} for t in new_structure['tables'] for c in t['cells']])
frame(actual)
frame(grids[:10])
print('원본 재직렬화 재현 성공 위치:',len(actual))

### 표 구조 개선 결과

원본 표의 행·열 구조를 복원하고 개선 전후 데이터 비교

- 표 내부에 포함된 제목, 본문, 항목 구분 확인
- 행(row), 열(column), 병합 정보(rowspan, colspan) 추출
- 원본 표의 셀 단위 텍스트 확인
- 개선 전후 텍스트 및 표 구조 비교
- 원본 제작셀 복원 성공 위치 3건 확인

표를 단순 텍스트로 처리하지 않고 셀 구조와 내용을 유지하도록 개선

## 7. 요구사항·메타데이터

저장된 승인 변경 원장 확인. 본문 ID와 소유 ID를 구분하고 승인 필드만 대조.

In [ ]:
from requirement_metadata import 포함, 허용필드
meta_changes=read(data_versions()/'version_04_requirements_20261007T015054_7e360551/요구사항metadata_변경내역.json')
frame([{'record_id':r['record_id'],'변경 필드':list(r['변경']),'전':clip({k:v['전'] for k,v in r['변경'].items()},800),'후':clip({k:v['후'] for k,v in r['변경'].items()},800)} for r in meta_changes[:3]])
assert all(set(r['변경'])<=허용필드 for r in meta_changes)
print('저장된 metadata 변경 records:',len(meta_changes))

### 요구사항 메타데이터 보정

원본 문서의 요구사항 ID와 관련 정보를 확인하고 메타데이터 보완

- 요구사항 ID(`requirement_ids`) 추가 및 수정
- 요구사항 별칭(`requirement_aliases`) 연결
- 원본 요구사항 정보(`source_requirement_fields`) 보완
- 기존 본문 ID와 요구사항 ID를 구분하여 관리
- 메타데이터 변경 기록 337건 확인

승인된 필드만 수정하고, 변경 전후 데이터를 비교함.

In [ ]:
version=data_versions()/'version_04_requirements_20261007T015054_7e360551'
rows={r['record_id']:r for r in jl(version/'rag_input.jsonl')}
examples=[]
for change in meta_changes[:3]:
    r=rows[change['record_id']]
    assert all(r.get(k)==v['후'] for k,v in change['변경'].items())
    examples.append({'record_id':r['record_id'],'소유 요구사항':r['requirement_ids'],'본문 참조':r.get('referenced_requirement_ids'),'ID 본문 포함':{identity:포함(r['raw_text'],identity) for identity in r['requirement_ids']},'실제 본문':clip(r['raw_text']),'원본 필드 근거':clip(r.get('source_requirement_fields'),600)})
frame(examples)
frame([{'설정 문서':doc,'승인된 대응':pairs} for doc,pairs in read(LEGACY/'config/요구사항metadata_승인대응.json')['한정문서대응'].items()][:5])

### 요구사항 ID 연결 확인

요구사항 메타데이터 보정 후 실제 본문과 연결된 ID 확인

- 소유 요구사항(`requirement_ids`)과 본문에서 참조한 요구사항 ID 구분
- `SFR-상담-001`, `SFR-상담-002` 등 요구사항 ID 연결 확인
- 요구사항 ID가 실제 본문에 포함되어 있는지 검사
- 원본 필드 근거와 승인된 요구사항 ID 대응 관계 확인
- 문서별 승인된 ID 매핑 결과 확인

요구사항 ID를 본문과 연결하고, 원본 근거를 함께 보존함.


## 8. 중간 정제 버전

여기부터는 승인된 기존 단계별 결과를 입력으로 사용. 누락된 승인 패킷 때문에 5단계 결과에서 최종 records까지 자동 재생할 수 없다.

In [ ]:
versions=[
('이번 최초 재실행',Path(state()['initial'])/'rag_input.jsonl'),
('이번 문자 복구',Path(state()['repaired'])/'rag_input.jsonl'),
('10/7 frozen',PROJECT/'최종공유용/data/rag_input_final.jsonl'),
('Stage8_2N',HISTORY/'outputs/stage8_2r_final_v1/data/rag_input_stage8_2n_experiment.jsonl'),
('Stage10_2',HISTORY/'outputs/stage10_2_structural_candidate_v1/improved_records.jsonl'),
('Stage11 hybrid',HISTORY/'outputs/stage11_hybrid_dataset/hybrid_records.jsonl'),
('최종',FINAL/'final_records.jsonl')]
summary=[]
for name,path in versions:
    count=0;docs=set();fields=Counter()
    for r in jl(path):
        count+=1;docs.add(r.get('sha256',r.get('doc_id')));fields.update(r.keys())
    summary.append({'단계':name,'records':count,'문서':len(docs),'metadata 필드':len(fields),'파일':str(path)})
frame(summary)
print('서로 다른 버전의 record 수가 같다고 같은 데이터라고 판정하지 않습니다.')

### 중간 정제 데이터 버전 비교

전처리 단계별로 저장된 데이터의 레코드 수와 메타데이터 필드 확인

- 최초 재실행: 22,435개 레코드 / 메타데이터 31개 필드
- 문자 복구 후: 22,435개 / 35개 필드
- 10/7 확정본: 21,701개 / 72개 필드
- Stage8: 21,902개 / 85개 필드
- Stage10: 21,902개 / 87개 필드
- Stage11: 21,902개 / 88개 필드
- 최종: 21,902개 / 89개 필드

모든 단계에서 문서 수는 98개로 동일함.

전처리 과정에서 레코드 수와 메타데이터 구성이 변경되었으며, 최종적으로 21,902개 레코드와 89개 메타데이터 필드가 확인됨.

## 9. Elements (문서를 구성하는 요소) 생성

기존 Stage8 records를 map_records로 변환. 최종 Patches 입력은 Stage11 baseline으로 별도 연결.

In [ ]:
REMAP = True  # True면 기존 mapper로 전체 Stage8 records를 새 경로에 변환
input_path=HISTORY/'outputs/stage8_2r_final_v1/data/rag_input_stage8_2n_experiment.jsonl'
if REMAP:
    mapped=unique_output('07_mapped_elements.jsonl')
    command(SETTINGS['chunk_python'],BASE/'replay_mapping.py',['--input',input_path,'--output',mapped],'07_mapper')
    save_state({'mapped':mapped})
mapped=Path(state()['mapped'])
reference=HISTORY/'work/stage8_2QA/native/stage8_2n_evaluation/elements.jsonl'
frame([checks_equal_jsonl(mapped,reference)])
frame([{'element_id':e['element_id'],'kind':e['kind'],'검색표현 기반 본문':clip(e['raw_text']),'table':clip(e.get('table'),650),'location':clip(e['location'],650)} for e in list(jl(mapped))[:3]])

### Elements 생성 결과

Stage8 정제 레코드를 Elements 형식으로 변환

- 대상 문서: 98개
- 생성된 Elements: 25,996개
- 매핑된 레코드: 21,146개
- Paragraph 요소: 11,964개
- 요구사항 ID가 포함된 레코드: 5,278개
- 기존 Native Elements와 비교 결과 일치
- 전체 값 및 바이너리 해시 동일 확인

기존 Stage8 데이터를 Elements로 변환하고, 원본 레코드 위치와 표 정보를 연결함.

## 10. Patches(기존 데이터를 일부 수정하는 작업) 적용

Stage11 baseline의 before 값을 대조하고 after로 교체. Stage8 mapper 출력에 바로 적용하는 패치가 아니다.

In [ ]:
REPATCH = True
if REPATCH:
    element_output=unique_output('07_final_elements.jsonl')
    command(SETTINGS['legacy_python'],FINAL/'adapter/reproduce_elements.py',['--output',element_output],'07_patches')
    save_state({'elements':element_output})
frame([checks_equal_jsonl(state()['elements'],FINAL/'final_elements.jsonl')])
patches=list(jl(FINAL/'adapter/patches.jsonl'))
frame([{'element_id':p['element_id'],'전 kind':p['before']['kind'],'후 kind':[e['kind'] for e in p['after']],'전 table':clip(p['before'].get('table'),700),'후 table':clip([e.get('table') for e in p['after']],700),'텍스트 유지':len(p['after'])==1 and p['before']['raw_text']==p['after'][0]['raw_text']} for p in patches[:3]])
print('확인된 최종 patch 수:',len(patches))

### Patches 적용 및 최종 Elements 생성

Stage11 기준 데이터에 승인된 Patch를 적용하여 최종 Elements 생성

- 최종 Elements: 26,023개
- 적용된 Patch: 4,909개
- 표 구조 및 요소 정보 보완
- 기존 최종 데이터와 전체 값 및 해시 일치 확인

기존 전처리 결과를 재생성하고 최종 데이터와 동일한지 검증함.

## 11. 전처리 결과 검증

최종 records는 기존 파일과 manifest 대조. 새 Elements는 전체 값·순서·해시 비교. 원본 의미 완전성은 미검증.

초기·문자 복구 결과도 기존 재실행본과 비교. 문자 복구의 이전 버전 참조는 실행 폴더 차이만 허용.

In [ ]:
manifest = read(FINAL / 'manifest.json')
# 전처리 산출물과 재생 입력만 검사한다.
selected_files = ['final_records.jsonl', 'final_elements.jsonl',
                  'adapter/baseline_stage11_elements.jsonl', 'adapter/patches.jsonl']
for rel in selected_files:
    assert sha(FINAL / rel) == manifest['files'][rel]['sha256'], rel
records = list(jl(FINAL / 'final_records.jsonl'))
elements = list(jl(state()['elements']))
counts = {'documents': len({r['sha256'] for r in records}),
          'records': len(records), 'elements': len(elements)}
assert all(value == manifest[key] for key, value in counts.items())
record_ids = {r['record_id'] for r in records}
assert len(record_ids) == len(records)
assert len({e['element_id'] for e in elements}) == len(elements)
assert {r['sha256'] for r in records} == {d['sha256'] for d in documents}
assert all(e['location']['record_id'] in record_ids for e in elements)
frame([{'항목': k, '기존': manifest[k], '이번': v} for k, v in counts.items()])
frame([checks_equal_jsonl(state()['elements'], FINAL / 'final_elements.jsonl')])
frame([{'record_id': r['record_id'], '정제문': clip(r['clean_text']),
        '요구사항': r['requirement_ids'], '출처': r['source_paths']} for r in records[:3]])
# 최종 records는 기존 파일 검증, Elements는 새로 재생한 결과 비교.
verification = {'counts': counts, 'elements_equal': checks_equal_jsonl(state()['elements'], FINAL / 'final_elements.jsonl'),
                'records_manifest_match': True, 'whole_raw_to_final_reproduced': False,
                'run_paths': {k: state()[k] for k in ['initial', 'repaired', 'rechecked', 'mapped', 'elements']}}
report = unique_output('validation.json')
report.write_text(json.dumps(verification, ensure_ascii=False, indent=2))
print('검증 기록:', report)

### 전처리 결과 검증

최종 전처리 데이터와 기존 결과를 비교하여 검증

- 문서: 98개 → 98개
- Records: 21,902개 → 21,902개
- Elements: 26,023개 → 26,023개
- 최종 Elements 전체 값 일치: True
- 최종 Elements 바이트 해시 일치: True

파일 해시, 데이터 개수, ID 중복 여부 및 Records와 Elements 간 연결 관계를 검증함.

기존 최종 결과와 동일한 데이터가 생성된 것을 확인함.

In [ ]:
reference_state = read(BASE / 'run_state.json')
comparison = []
for stage in ['initial', 'repaired']:
    current = Path(state()[stage]) / 'rag_input.jsonl'
    reference = Path(reference_state[stage]) / 'rag_input.jsonl'
    path_changes = 0
    count = 0
    for count, (left, right) in enumerate(zip_longest(jl(current), jl(reference)), 1):
        assert left is not None and right is not None
        left, right = dict(left), dict(right)
        if stage == 'repaired':
            # 실행 폴더만 다르다. 이전 버전 참조는 상대 파일까지 대조한다.
            left_ref = left.pop('previous_version_reference', None)
            right_ref = right.pop('previous_version_reference', None)
            if left_ref != right_ref:
                assert left_ref and right_ref
                right_path = Path(right_ref)
                if not right_path.is_absolute():
                    right_path = Path(reference_state['initial']).parents[1] / right_path
                assert Path(left_ref).relative_to(Path(state()['initial'])) == right_path.relative_to(Path(reference_state['initial']))
                path_changes += 1
            else:
                assert left_ref == right_ref
        assert left == right, f'{stage}: 행 {count} 불일치'
    comparison.append({'단계': stage, 'records': count, '본문·좌표·메타데이터 일치': True,
                       '실행 경로만 변경': path_changes, '파일 해시 동일': sha(current) == sha(reference)})
# 전체 원본 추출 요소도 문서별 값 비교.
raw_count = 0
for current in sorted((Path(state()['initial']) / 'raw').glob('*.json')):
    reference = Path(reference_state['initial']) / 'raw' / current.name
    assert read(current) == read(reference)
    raw_count += len(read(current))
frame(comparison)
print('기존 원본 추출 요소와 일치:', raw_count)


### 초기 추출 및 문자 복구 결과 검증

기존 실행 결과와 새로 생성한 데이터를 비교하여 일치 여부 확인

- 초기 정제(initial): 22,435개 Records
- 문자 복구(repaired): 22,435개 Records
- 본문·좌표·메타데이터 일치: 모두 True
- 초기 정제 파일 해시 일치: True
- 문자 복구 파일 해시 일치: False
- 문자 복구 단계 실행 경로 변경: 389건
- 기존 원본 추출 요소 13,010개 일치 확인

문자 복구 단계의 파일 해시는 실행 경로 차이로 일치하지 않았으나, 본문·좌표·메타데이터는 기존 결과와 일치함.